In [1]:
%pip install -r ../requirements.txt
%pip install -q sentence-transformers
%pip install -q hdbscan
%pip install -q umap-learn
%pip install -q "nbformat>=4.2.0"

   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 625.0/625.0 kB 4.4 MB/s  0:00:00
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 596.4/596.4 kB 6.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 18.2 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 26.4 MB/s  0:00:00
   ---------------------------------------- 0.0/43.0 MB ? eta -:--:--
   ---------------- ----------------------- 17.3/43.0 MB 86.3 MB/s eta 0:00:01
   -------------------------------- ------- 34.9/43.0 MB 83.5 MB/s eta 0:00:01
   ---------------------------------------- 43.0/43.0 MB 69.5 MB/s  0:00:00

   ---------------------------------------- 0/7 [llvmlite]
   ---------------------------------------- 0/7 [llvml


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import dotenv
import os

In [3]:
dotenv.load_dotenv()

CHALLENGES_PATH = '../output/clean_challenges.csv'
EXPECTATIONS_PATH = '../output/clean_expectations.csv'

df = pd.read_csv(CHALLENGES_PATH)
expectations_df = pd.read_csv(EXPECTATIONS_PATH)

TEXT_COLUMN = "pain_points"

df = df.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)

In [ ]:
df.head(5)

,focus_group,pain_points,sentiment
0,CPO Central Permit Office,Split Permit & Code Systems: Permit and Code E...,negative
1,CPO Central Permit Office,Scattered Information: Property and permit inf...,negative
2,CPO Central Permit Office,Research Complexity: Users must search multipl...,negative
3,CPO Central Permit Office,Difficult Information Retrieval: Finding relev...,negative
4,CPO Central Permit Office,Limited IPS-Camino Integration: Information do...,negative


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")

challenge_embeddings = model.encode(
    df["pain_points"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = challenge_embeddings

In [ ]:
# add embeddings to the dataframe
df["embeddings"] = embeddings.tolist()

In [ ]:
df.head(5)

In [ ]:
def assign_category(text, category_config):
    text_lower = str(text).lower()
    best_category = "Other / Uncategorized"
    best_score = 0

    for category, keywords in category_config.items():
        score = 0
        for kw in keywords:
            if kw in text_lower:
                # Weight longer phrases a bit higher than single keywords
                score += max(1, len(kw.split()))

        if score > best_score:
            best_score = score
            best_category = category

    return best_category

# Categorize each challenge before clustering
df["raw_category"] = df[TEXT_COLUMN].fillna("").apply(
    lambda value: assign_category(value, CATEGORY_CONFIG)
)

df[[TEXT_COLUMN, "raw_category"]].head()

In [ ]:
df[df["raw_category"] == "Other / Uncategorized"].head(5)

In [ ]:
print(f"Total records: {df.shape[0]}")
df[[TEXT_COLUMN, "raw_category"]].groupby("raw_category").size().sort_values(ascending=False)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

clusterer = AgglomerativeClustering(
    n_clusters=12
)

labels = clusterer.fit_predict(embeddings)

In [ ]:
#create catergories out of the pain points and assign them to the dataframe


In [ ]:
import umap
import hdbscan
from sklearn.metrics import silhouette_score

# Reduce embedding space before density clustering for more stable groups
reducer_for_cluster = umap.UMAP(
    n_neighbors=20,
    n_components=15,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)
reduced_embeddings = reducer_for_cluster.fit_transform(embeddings)

candidate_min_cluster_sizes = [5, 8, 10, 12, 15]
results = []

best_model = None
best_labels = None
best_score = -1.0

for mcs in candidate_min_cluster_sizes:
    model = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=max(2, mcs // 2),
        metric="euclidean",
        cluster_selection_method="eom"
    )
    labels_candidate = model.fit_predict(reduced_embeddings)

    clustered_mask = labels_candidate != -1
    n_clusters = len(set(labels_candidate[clustered_mask]))

    if clustered_mask.sum() > 1 and n_clusters > 1:
        score = silhouette_score(
            reduced_embeddings[clustered_mask],
            labels_candidate[clustered_mask],
            metric="euclidean"
        )
    else:
        score = -1.0

    results.append({
        "min_cluster_size": mcs,
        "clusters_found": n_clusters,
        "noise_points": int((labels_candidate == -1).sum()),
        "silhouette": round(float(score), 4)
    })

    if score > best_score:
        best_score = score
        best_model = model
        best_labels = labels_candidate

quality_df = pd.DataFrame(results).sort_values("silhouette", ascending=False)
display(quality_df)

print(f"Chosen silhouette: {best_score:.4f}")

In [ ]:
df["Cluster"] = best_labels

n_clusters = df.loc[df["Cluster"] != -1, "Cluster"].nunique()
noise_points = int((df["Cluster"] == -1).sum())

print(f"Clusters found: {n_clusters}")
print(f"Unclustered/noise rows: {noise_points}")

In [ ]:
for cluster in sorted(df["Cluster"].unique()):

    print("=" * 80)
    print(f"Cluster {cluster}")

    sample = df[df["Cluster"] == cluster]["pain_points"].head(5)

    for text in sample:
        print("-", text)

In [ ]:
# 2D semantic map of challenges by cluster
import umap
import plotly.express as px

viz_reducer = umap.UMAP(
    n_neighbors=20,
    min_dist=0.05,
    metric="cosine",
    random_state=42
)
coords = viz_reducer.fit_transform(embeddings)

plot_df = pd.DataFrame({
    "x": coords[:, 0],
    "y": coords[:, 1],
    "Cluster": df["Cluster"].astype(str),
    "Challenge": df["pain_points"],
    "Focus Group": df["focus_group"]
})

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="Cluster",
    hover_data=["Focus Group", "Challenge"],
    title="Semantic Cluster Map (UMAP)"
 )
fig.update_layout(xaxis_title="UMAP 1", yaxis_title="UMAP 2")
fig.show()

# Cluster size view: x = number of records, y = cluster number
cluster_counts = (
    df[df["Cluster"] != -1]
    .groupby("Cluster", as_index=False)
    .size()
    .rename(columns={"size": "Number of records in that Cluster"})
)
cluster_counts["Cluster"] = cluster_counts["Cluster"].astype(int)

fig_counts = px.bar(
    cluster_counts.sort_values("Cluster"),
    x="Number of records in that Cluster",
    y="Cluster",
    orientation="h",
    color="Cluster",
    title="Cluster Size by Cluster Number"
 )
fig_counts.update_layout(
    xaxis_title="Number of records in that Cluster",
    yaxis_title="Cluster Number"
 )
fig_counts.show()

In [ ]:
cluster0 = df[df.Cluster == 0].copy()

cluster0_embeddings = embeddings[df.Cluster == 0]

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    prediction_data=True
)

subclusters = clusterer.fit_predict(cluster0_embeddings)

cluster0["Subcluster"] = subclusters

In [ ]:
# show unclustered data (if any)
unclustered = df[df["Cluster"].isna() | (df["Cluster"] == -1)]

print(f"Unclustered rows: {len(unclustered)}")

if unclustered.empty:
    print("No unclustered data found.")
else:
    display(unclustered[["focus_group", "pain_points", "Cluster"]].head(20))


In [ ]:
#show clusters
clustered_data = df[df["Cluster"].notna() & df["Cluster"] != -1]

print(f"Cluster : {clustered_data[clustered_data['Cluster'] == 8].shape[0]}")
print(f"Cluster data: {clustered_data[clustered_data['Cluster'] == 8].head()}")


In [ ]:
cluster_names = {
    0: "",
    1: "Ticketing & Workflow",
    2: "Multiple System",
    3: "Visibility & Transparency",
    4: "Tracking & Reporting",
    5: "Research Complexity",
    6: "UI / UX & Navigation",
    7: "Scheduling",
    8: "",
    9: "",
    10: "",
    11: "",
    12: "",
    13: "",
    14: "",
    15: "",
    25: "",
    # ...
}

# Keep the pre-clustering Category column and store cluster naming separately
df["Cluster_Label"] = df["Cluster"].map(cluster_names)